In [ ]:
# Load shared paths from the repo-root config.py
import sys, os
sys.path.insert(0, os.path.abspath("../.."))   # repo root (run this notebook from its experiment folder)
import config


In [1]:
from model import *
from evaluation_helper import evaluate_full_pipeline
import pandas as pd
test_data = pd.read_parquet(config.TECHNIQUE_TEST_PARQUET)

# IMPORTANT (same fix as training)
test_data["techniques"] = test_data["techniques"].apply(parse_labels)
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# rebuild label map (same as training)
label2id, id2label = build_label_map(test_data)

model = TechniqueClassifier(CFG.model_name, len(label2id))
checkpoint = torch.load("best_technique_model.pt")

model.load_state_dict(checkpoint["model_state_dict"], strict=False)

label2id = checkpoint["label2id"]
id2label = checkpoint["id2label"]
model.to(CFG.device)
evaluate_full_pipeline(
    model=model,
    test_df=test_data,
    dataset_class=TechniqueDataset,
    tokenizer=tokenizer,
    label2id=label2id,
    id2label=id2label,
    scorer_script_path="task-TC_scorer.py",
    techniques_list_path="propaganda-techniques-names-semeval2020task11.txt",
    output_dir="eval_results",
    device=CFG.device
)

/home/omer_ahmed/Experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total labels: 14


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 6346.13it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Gold file saved to: eval_results/gold.tsv


Predicting: 100%|██████████| 208/208 [00:36<00:00,  5.68it/s]


✅ Submission file saved to: eval_results/submission.tsv

🚀 Running SemEval scorer...


🚀 RELAXED SCORING RESULTS
Precision=0.884317
Recall=0.891226
F1=0.887758

📊 Per-class F1:
F1_Appeal_to_Authority=0.727272
F1_Appeal_to_fear-prejudice=0.843537
F1_Bandwagon,Reductio_ad_hitlerum=0.733333
F1_Black-and-White_Fallacy=0.842105
F1_Causal_Oversimplification=0.882352
F1_Doubt=0.897260
F1_Exaggeration,Minimisation=0.882352
F1_Flag-Waving=0.897959
F1_Loaded_Language=0.933450
F1_Name_Calling,Labeling=0.921108
F1_Repetition=0.852071
F1_Slogans=0.828282
F1_Thought-terminating_Cliches=0.769230
F1_Whataboutism,Straw_Men,Red_Herring=0.597402



In [ ]:
import torch
ckpt = torch.load("best_span_roberta_pos_ner_discourse_updated.pt")

print(ckpt["cfg"])

{'model_name': 'roberta-large', 'max_length': 512, 'stride': 384, 'batch_size': 2, 'lr': 2e-05, 'weight_decay': 0.01, 'epochs': 5, 'warmup_ratio': 0.1, 'num_workers': 2, 'seed': 42, 'lstm_hidden': 512, 'lstm_layers': 1, 'pos_dim': 32, 'ner_dim': 32, 'discourse_dim': 16, 'discourse_input_dim': 11, 'dropout': 0.2, 'save_path': 'best_span_roberta_pos_ner_discourse.pt', 'class_weights': (1.0, 1.0, 1.0, 1.0, 1.0), 'ce_loss_weight': 0.0, 'crf_loss_weight': 1.0, 'min_token_overlap_ratio': 0.0, 'pos_scale': 0.3, 'ner_scale': 0.3, 'discourse_scale': 0.3}
